In [ ]:
import pandas as pd
import geopandas as gpd
import glob
from shapely.geometry import Point

# Path to the folder containing the CSV files
folder_path = './bikeshare_data/sf_bikeshare'
# Use glob to find all CSV files in the folder
csv_files = glob.glob(f"{folder_path}/*.csv")

df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
df.columns

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 0. constants
SHAPEFILE = "./tract_shapefiles/tl_2023_ca_tract/tl_2023_06_tract.shp"
OUT_CSV   = "./bikeshare_data/tracts_with_trips/sf_tracts_with_trips.csv"
NUM_DAYS  = 821

# 1. build one stations table from both start/end
stations = pd.concat([
    df[["start_station_id","start_lat","start_lng","start_station_name"]]
      .rename(columns={
         "start_station_id": "Station ID",
         "start_lat":        "Station Latitude",
         "start_lng":        "Station Longitude",
         "start_station_name": "Station Name"
      }),
    df[["end_station_id","end_lat","end_lng","end_station_name"]]
      .rename(columns={
         "end_station_id":   "Station ID",
         "end_lat":          "Station Latitude",
         "end_lng":          "Station Longitude",
         "end_station_name": "Station Name"
      })
], ignore_index=True)

# 2. collapse duplicates by taking median coords + first name
station_info = (
    stations
      .groupby("Station ID", as_index=False)
      .agg({
        "Station Latitude": "median",
        "Station Longitude":"median",
        "Station Name":     "first"
      })
)

# 3. count trips out + in via value_counts
trips = pd.concat([
    df["start_station_id"].value_counts().rename("trips_out"),
    df["end_station_id"].value_counts().rename("trips_in")
], axis=1).fillna(0)
trips["total_trips"] = trips["trips_out"] + trips["trips_in"]
trips = trips.reset_index().rename(columns={"index":"Station ID"})

# 4. make a GeoDataFrame of stations + join to tracts
stations_gdf = (
    station_info
      .merge(trips[["Station ID","total_trips"]], on="Station ID", how="left")
)
stations_gdf["geometry"] = Point  # placeholder; we’ll vectorize next

# vectorized point creation
stations_gdf = gpd.GeoDataFrame(
    stations_gdf,
    geometry=gpd.points_from_xy(
      stations_gdf["Station Longitude"],
      stations_gdf["Station Latitude"]
    ),
    crs="EPSG:4326"
)

tracts = gpd.read_file(SHAPEFILE).to_crs(stations_gdf.crs)
stations_with_tracts = gpd.sjoin(
    stations_gdf,
    tracts[["GEOID","ALAND","geometry"]],
    how="left",
    predicate="within"
).rename(columns={"Station ID":"id"})

# 5. aggregate by tract
tracts_with_trips = (
    stations_with_tracts
      .groupby("GEOID", as_index=False)
      .agg({
        "total_trips": "sum",
        "ALAND":       "first"
      })
)

# 6. normalize to trips/day & write out
tracts_with_trips["trips_per_day"] = tracts_with_trips["total_trips"] / NUM_DAYS
tracts_with_trips.to_csv(OUT_CSV, index=False)

print("Columns:", tracts_with_trips.columns.tolist())
